<a href="https://colab.research.google.com/github/ErickJester/expo-escom/blob/main/00_conteo_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ Herramientas de Dataset — ExpoEscom
Menú con utilidades sobre una carpeta de Drive (incluye "Compartido conmigo").

**Opciones del menú:**
1. **Contabilizar** — cuenta las imágenes de una carpeta (recursivo).
2. **Aplanar** — saca las imágenes de una subcarpeta mal anidada y las sube
   a la carpeta superior (las deja donde deberían estar).

**Cómo usar:** corre las celdas 1→4 una vez (setup), luego ejecuta la última
celda (Menú) cuantas veces quieras.

In [ ]:
VERSION = '3.1.0'

print('═' * 50)
print('🛠️  Herramientas de Dataset — ExpoEscom')
print(f'v{VERSION}')
print('═' * 50)

from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
service = build('drive', 'v3')
print('✅ Autenticado con Drive API')

In [ ]:
# ════ ÚNICA CONFIGURACIÓN NECESARIA ══════════════════════════

FOLDER_ID = '1YwNZMW67NYGMb_g-74kG5gb2u_m_3c1J'

EXTENSIONES_IMAGEN = {'.jpg'}
TARGET_IMAGENES    = 100_000
print(f'Folder ID : {FOLDER_ID}')
print(f'Extensión : solo .jpg  |  Meta: {TARGET_IMAGENES:,} imágenes')

In [ ]:
# ════ HELPERS DE DRIVE API ════════════════════════════════════
from pathlib import Path

_ARGS = dict(supportsAllDrives=True, includeItemsFromAllDrives=True,
             pageSize=1000)
_FOLDER_MIME = 'application/vnd.google-apps.folder'


def _es_imagen(f):
    # Solo cuenta archivos con extensión .jpg (sin importar el mimeType)
    return Path(f['name']).suffix.lower() in EXTENSIONES_IMAGEN


def listar_contenido(parent_id):
    """Lee el nivel directo de parent_id.
    Devuelve (subcarpetas, n_imagenes_sueltas):
      subcarpetas = lista de dicts {id, name} ordenada por nombre.
      n_imagenes_sueltas = imágenes que cuelgan directo del nivel."""
    subcarpetas, sueltas, token = [], 0, None
    while True:
        resp = service.files().list(
            q=f"'{parent_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageToken=token, **_ARGS).execute()
        for f in resp.get('files', []):
            if f['mimeType'] == _FOLDER_MIME:
                subcarpetas.append({'id': f['id'], 'name': f['name']})
            elif _es_imagen(f):
                sueltas += 1
        token = resp.get('nextPageToken')
        if not token:
            break
    subcarpetas.sort(key=lambda c: c['name'].lower())
    return subcarpetas, sueltas


def contar_imagenes(folder_id):
    """Cuenta recursivamente los archivos .jpg bajo folder_id."""
    total, stack = 0, [folder_id]
    while stack:
        fid, token = stack.pop(), None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType)',
                pageToken=token, **_ARGS).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == _FOLDER_MIME:
                    stack.append(f['id'])
                elif _es_imagen(f):
                    total += 1
            token = resp.get('nextPageToken')
            if not token:
                break
    return total


def mover_contenido(origen_id, destino_id):
    """Mueve TODOS los hijos directos de origen_id hacia destino_id.
    En Drive 'mover' es solo cambiar el parent: no re-sube nada.
    Devuelve conteos {imagenes, carpetas, otros}."""
    # 1) recolectar todo primero (mover muta el parent, no paginar a la vez)
    items, token = [], None
    while True:
        resp = service.files().list(
            q=f"'{origen_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageToken=token, **_ARGS).execute()
        items.extend(resp.get('files', []))
        token = resp.get('nextPageToken')
        if not token:
            break
    # 2) mover uno por uno
    movidos = {'imagenes': 0, 'carpetas': 0, 'otros': 0}
    for f in items:
        try:
            service.files().update(
                fileId=f['id'], addParents=destino_id,
                removeParents=origen_id, fields='id',
                supportsAllDrives=True).execute()
            if f['mimeType'] == _FOLDER_MIME:
                movidos['carpetas'] += 1
            elif _es_imagen(f):
                movidos['imagenes'] += 1
            else:
                movidos['otros'] += 1
        except Exception as e:
            print(f'   ⚠️ No se pudo mover {f["name"]}: {e}')
    return movidos


def construir_indice(parent_id, incluir_raiz=True):
    """Arma un índice numerado de las subcarpetas de parent_id.
    Si incluir_raiz: [0] representa la carpeta completa."""
    raiz = service.files().get(
        fileId=parent_id, fields='name', supportsAllDrives=True).execute()
    subs, sueltas = listar_contenido(parent_id)
    indice = {}
    if incluir_raiz:
        indice[0] = {'name': f'(TODA la carpeta «{raiz["name"]}»)',
                     'id': parent_id}
    for i, c in enumerate(subs, 1):
        indice[i] = c
    return raiz['name'], indice, sueltas


def mostrar_indice(indice, titulo='ÍNDICE DE CARPETAS'):
    print('═' * 50)
    print(titulo)
    print('═' * 50)
    for num, c in indice.items():
        print(f'  [{num:>2}]  {c["name"]}')
    print('═' * 50)


def pedir_opcion(indice, pregunta):
    """Pide un número válido del índice; reintenta hasta que sea correcto."""
    while True:
        sel = input(f'\n{pregunta} (número): ').strip()
        if sel.isdigit() and int(sel) in indice:
            return int(sel)
        print('   ⚠️  Número inválido, intenta de nuevo.')


print('✅ Helpers listos')

In [ ]:
# ════ OPCIONES DEL MENÚ ═══════════════════════════════════════

def opcion_contar():
    """[1] Contabilizar imágenes .jpg de una carpeta."""
    nombre, indice, sueltas = construir_indice(FOLDER_ID, incluir_raiz=True)
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    if sueltas:
        print(f'⚠️  {sueltas:,} .jpg sueltos en la raíz '
              f'(la opción [0] los incluye).')

    sel = pedir_opcion(indice, '¿Qué carpeta quieres contabilizar?')
    elegida = indice[sel]
    print(f'\n⏳ Contando .jpg en: {elegida["name"]}…')
    n = contar_imagenes(elegida['id'])

    faltan = max(0, TARGET_IMAGENES - n)
    pct    = n / TARGET_IMAGENES * 100

    print('\n' + '═' * 50)
    print(f'  Carpeta  : {elegida["name"]}')
    print(f'  .jpg     : {n:,}  ({pct:.1f}% de {TARGET_IMAGENES:,})')
    if faltan:
        print(f'  Faltan   : {faltan:,}')
    else:
        print(f'  ✅ Meta alcanzada ({TARGET_IMAGENES:,})')
    print('═' * 50)


def opcion_aplanar():
    """[2] Subir el contenido de una subcarpeta mal anidada a su carpeta padre."""
    # 1) elegir la carpeta donde están las imágenes mal anidadas
    nombre, indice, _ = construir_indice(FOLDER_ID, incluir_raiz=True)
    print(f'\n📂 Carpeta raíz : {nombre}')
    mostrar_indice(indice)
    sel = pedir_opcion(indice, '¿En qué carpeta están las imágenes mal anidadas?')
    destino = indice[sel]

    # 2) buscar subcarpetas dentro de la carpeta elegida
    subs, _ = listar_contenido(destino['id'])
    if not subs:
        print(f'\n✅ «{destino["name"]}» no tiene subcarpetas. Nada que aplanar.')
        return

    subindice = {i: c for i, c in enumerate(subs, 1)}
    mostrar_indice(subindice,
                   titulo=f'SUBCARPETAS DENTRO DE «{destino["name"]}»')
    sub_sel = pedir_opcion(
        subindice, '¿Qué subcarpeta quieres vaciar (subir su contenido)?')
    subcarpeta = subindice[sub_sel]

    # 3) confirmar (operación que modifica tu Drive)
    print('\nSe moverá TODO el contenido de:')
    print(f'   «{subcarpeta["name"]}»   →   «{destino["name"]}»')
    if input('¿Confirmas? (si/no): ').strip().lower() not in ('si', 's', 'sí'):
        print('Cancelado. No se movió nada.')
        return

    # 4) mover
    print('\n⏳ Moviendo…')
    movidos = mover_contenido(subcarpeta['id'], destino['id'])
    total = sum(movidos.values())
    print('\n' + '═' * 50)
    print(f'  Movidos a «{destino["name"]}» : {total:,} elementos')
    print(f'    · .jpg     : {movidos["imagenes"]:,}')
    if movidos['carpetas']:
        print(f'    · carpetas : {movidos["carpetas"]:,}')
    if movidos['otros']:
        print(f'    · otros    : {movidos["otros"]:,}')
    print('═' * 50)

    # 5) opcional: enviar a la papelera la subcarpeta ya vacía
    resp = input(f'\n¿Eliminar la subcarpeta ya vacía «{subcarpeta["name"]}»? '
                 f'(si/no): ').strip().lower()
    if resp in ('si', 's', 'sí'):
        service.files().update(fileId=subcarpeta['id'], body={'trashed': True},
                               supportsAllDrives=True).execute()
        print('   🗑️  Subcarpeta enviada a la papelera.')
    else:
        print('   La subcarpeta vacía se conservó.')


print('✅ Opciones del menú listas')

## ▶️ Menú — ejecuta esta celda

In [ ]:
# ════ MENÚ PRINCIPAL ══════════════════════════════════════════
print('═' * 50)
print('  MENÚ PRINCIPAL')
print('═' * 50)
print('  [1]  Contabilizar imágenes de una carpeta')
print('  [2]  Aplanar: subir el contenido de una subcarpeta')
print('═' * 50)

_op = input('Elige una opción (1/2): ').strip()
if _op == '1':
    opcion_contar()
elif _op == '2':
    opcion_aplanar()
else:
    print('Opción inválida. Vuelve a ejecutar esta celda.')